# 手順3c：最終候補のモデルを MCMC で推定（新データ）

CLAUDE.md §5・§7 の手順3c。手順3b の最尤推定（ハイパーパラメータを 1 点に決める）で選んだ**最終候補のモデル**だけを、MCMC（自作の適応型ランダムウォーク・メトロポリス法。手順2と同じ）でも推定し、最尤推定と比べる。

- **何を確かめるか**：ハイパーパラメータ（振幅・長さスケール・ノイズ・線形項）の不確かさを予測に含めたとき、予測区間の広さ・被覆率・NLPD が最尤推定とどれだけ違うか。ハイパーパラメータが下限・上限に張り付いていないか。設定温度の効果（190/230 ℃ の粘度比）の不確かさ。
- **候補のモデル**：材料ごとに、B0T（ショットごとの γ̇）・R3（条件平均の γ̇）・R2（設定値だけ）の 3 つ。Tff・P3_2 を足したモデルは、手順3b で要らないと分かったので入れない。設定エリアで変えられる（手順3b の NLPD が良いモデルを足す設定もある）。
- **分割**：S_V（5data → 3data）と S_T（5data の 190・230 ℃ → 210 ℃）。
- **GPR**：ARD Matern 5/2 ＋ 線形カーネル（DotProduct）＋ ノイズ（手順3b と同じ）。標準化と中心化は学習データだけで計算する。
- **事前分布**：カーネルの探索範囲（`nb_common.py` の `kernel_for`）と同じ範囲の、対数スケールの一様分布（振幅² 1e-3〜1e3、長さスケール 1e-2〜1e2、線形項の振幅² 1e-3〜1e3、σ₀ 1e-5〜1e3、ノイズ分散 NOISE_LB〜10）。事後分布は範囲内で exp(対数周辺尤度) に比例するので、最頻値は最尤推定と一致する。
- **予測**：事後サンプルを等間隔に N_PRED_DRAWS 個選び、各サンプルでの GPR の予測分布（ノイズ込みの正規分布）を等しい重みで混ぜた混合分布を使う。

### 自作の適応型ランダムウォーク・メトロポリス法

1. 今いる点 θ（ハイパーパラメータの対数）の近くに、正規分布 N(θ, λ·Σ) から候補を出す。
2. 候補の対数事後密度（範囲内なら対数周辺尤度、範囲外なら −∞）を計算し、確率 min(1, exp(候補 − 現在)) で移動する。移動しなければその場にとどまる。
3. **調整期間**：前半 75% では、それまでの足跡から提案のばらつき Σ（パラメータどうしの相関も含む）を学ぶ。調整期間の間ずっと、採択率が 0.234 に近づくよう大きさ λ を合わせる。調整期間のサンプルは捨てる。
4. **本番**：Σ と λ を固定した、ふつうのメトロポリス法として歩く。R-hat が 1.01 を超えるか、有効サンプル数（ESS）が 400 に届かないときは、本番の回数を増やして、サンプリングし直す（最大 N_SAMP_MAX 回）。
5. 4 本のチェーンを別々の初期値から並列に回し、R-hat と有効サンプル数（ESS）で収束を確かめる。

## 使い方（このノートを一人で実行して、結果を判断するために）

1. メニューの「すべて実行」で、上から順に実行する（約 40 分）。カーネルは、このプロジェクトの `.venv` を選ぶ。
2. 設定を変えたいときは、最初の「設定エリア」のセルだけを直して、最初から実行し直す（高解像度のグラフの設定は、下の 7）。
3. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」。基準は、判定の文の中に書いてある。★ が出ても、エラーではない。説明書の「判定の読み方」で、★ の意味を確かめる。
4. 図と表は `results/step3c_mcmc/` に保存される。図の見方は、説明書 `results/学習結果説明_手順3c.pdf` にある。
5. データを取り直したとき、問題のあるデータを入れ替えたときは、`問題データ一覧.md` の手順に従う。このノートは、全条件のファイルを読み、`02_機械学習データ/問題データ一覧.csv` の「除外」を除く。
6. 用語や図の見方で分からないところは、説明書の用語集で調べる。記号（Tff、Tset、R3、E_h など）の意味は、`results/学習結果説明_記号表.pdf` に一覧がある。
7. **高解像度のグラフ**（報告書・発表資料用。GPR6月ノートのセル9-2 と同じ見せ方）は、「高解像度のグラフ」の節のセルで作る。**このセルだけは、セルの先頭にも設定エリアがあり、そこだけを直して、このセルだけを実行し直せる**（学習はやり直さない。全体を一度実行した後で）。保存先は `results/step3c_mcmc/highres/`（Git には入れない）。
8. サンプリングには時間がかかる。`USE_SAVED_SAMPLES = True` のときは、保存済みの事後サンプル（`results/step3c_mcmc/posterior_*.csv`）があれば読み込み、サンプリングを省く。データを取り直したときは `False` にする（`rerun_all.py` は、実行の前に保存済みのサンプルを消すので、そのままでよい。ノートだけを実行し直すときは、自分で `False` にするか、`posterior_*.csv` を消す）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
SEED = 42                     # 乱数シード
NOISE_LB = 1e-6               # ノイズ分散の下限（旧ノートは 1e-3）
N_RESTARTS = 5                # GPR の最適化のやり直し回数（増やすと遅いが、安定する）
USE_EXCLUSIONS = True         # True：問題データ一覧の「除外」を使う／False：全条件を使う（取り直した後の確認用）
REPO = ".."                   # リポジトリ直下（このノートは 01_機械学習コードフォルダ にある）
FIG_DPI = 200                 # 保存する PNG の解像度
CANDIDATES = None             # 候補のモデル。None なら手順3b の結果から自動で選ぶ。手動なら {"PP": ["B0T", "R3"], "MIX": ["B0T", "R3", "R2"]} のように書く
CAND_FIXED = ["B0T", "R3", "R2"]   # 自動のとき、必ず入れるモデル（B0T：ショットごとの γ̇、R3：条件平均の γ̇、R2：設定値だけ。どれも Tset あり。Tff・P3_2 を足したモデルは、要らないと分かったので入れない）
N_CAND_EXTRA = 0              # 自動のとき、必ず入れるもののほかに、S_V の NLPD が良い順に選ぶ数（0 なら選ばない）
CAND_POOL = ["B0FT", "B2T", "R3F", "R3P"]   # 追加で選ぶ対象
N_WARM, N_SAMP, CHAINS = 3000, 8000, 4     # 調整の反復数（捨てる）、本番の反復数、チェーン数
N_SAMP_MAX = 20000            # R-hat が 1.01 を超えたときに増やす本番の回数の上限
N_JOBS = 12                   # 並列に回す数（チェーンと最尤推定。CPU の数に合わせる）
N_PRED_DRAWS = 200            # 予測に使う事後サンプル数（間引き後）
USE_SAVED_SAMPLES = True      # 保存済みの事後サンプルがあれば読み込む（サンプリングを省く）
MAIN_DATA = "なし"            # 主に使う外れ値の扱い
TEMP_HOLDOUT = 210            # S_T でテストにする設定温度 [℃]
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル1.5：ライブラリ、フォント、出力先
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MaxNLocator
from joblib import Parallel, delayed
from scipy.special import logsumexp
from scipy.stats import norm
import arviz as az   # 収束診断（R-hat・有効サンプル数）に使う

OUT_DIR = os.path.join(REPO, "results", "step3c_mcmc")
STEP3B_DIR = os.path.join(REPO, "results", "step3b")
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
print("Python:", sys.executable, "/", sys.version.split()[0])
print("ArviZ :", az.__version__)
print("出力先:", os.path.abspath(OUT_DIR))

### 共通の部品

データの読み込み、問題データ一覧の「除外」、モデル、評価、データの分け方の関数。中身は `04_補助スクリプト/nb_common.py` と同じ。

In [ ]:
# セル：共通の部品（読み込み・モデル・評価）。04_補助スクリプト/nb_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                 # ノイズ分散の下限（CLAUDE.md §5。旧ノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 5)                # GPR の最適化のやり直し回数
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))   # リポジトリ直下（ノートは 01_機械学習コードフォルダ にあるので ".."。スクリプトは環境変数 NB_REPO で指定）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)     # 問題データ一覧の「除外」を使うか
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限への張り付きは kernels.csv に記録する

# ---------------------------------------------------------------- データの場所
DATA_DIR = os.path.join(REPO, "02_機械学習データ", "NR500")
TEST_DIR = os.path.join(REPO, "03_予測検証用テストデータ", "NR500")
PROBLEM_CSV = os.path.join(REPO, "02_機械学習データ", "問題データ一覧.csv")
MATS = {"PP": "PP", "MIX": "粉砕材混合PP"}
FILES = {   # (材料, 組): (上位フォルダ, 中間フォルダ, フォルダ, ファイル名)。どれも全条件のファイル（問題のある条件は、一覧で除く）
    ("PP", "5data"): (DATA_DIR, "02_バージンPP再々実験_5data", "03_pp_max_5data", "結合_03_pp_max_5data_外れ値{o}.csv"),
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "02_pp_max_3data", "結合_02_pp_max_3data_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "01_pp_max_Alldata", "結合_01_pp_max_Alldata_外れ値{o}.csv"),
    ("MIX", "5data"): (DATA_DIR, "04_粉砕材混合PP実験_5data", "03_mixpp_max_5data", "結合_03_mixpp_max_5data_外れ値{o}.csv"),
    ("MIX", "3data"): (TEST_DIR, "04_粉砕材混合PP実験_5data用検証データ(3data)", "02_mixpp_max_3data", "結合_02_mixpp_max_3data_外れ値{o}.csv"),
    ("MIX", "Alldata"): (DATA_DIR, "03_粉砕材混合PP実験_alldata", "01_mixpp_max_Alldata", "結合_01_mixpp_max_Alldata_外れ値{o}.csv"),
}
LEGACY_FILES = {   # 1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）。一覧の「除外」で同じデータになるかの確認用
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "05_pp_max_3data(no1.5-3.0)", "結合_05_pp_max_3data(no1.5-3.0)_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "04_pp_max_Alldata(no1.5-3.0)", "結合_04_pp_max_Alldata(no1.5-3.0)_外れ値{o}.csv"),
}
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "加熱シリンダ温度[℃]": "Tset", "流動中樹脂温度(℃)": "Tff"}
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm\.csv$"
D = {}                  # (材料, 組, 外れ値) → DataFrame。ノートで D = load_all() として入れる
REMOVED = {}            # (材料, 組, 外れ値) → 一覧の「除外」で除いた行数


def path_of(mat, grp, o, files=None):
    top, d1, d2, fn = (files or FILES)[(mat, grp)]
    return os.path.join(top, d1, d2, f"外れ値{o}", fn.format(o=o))


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MATS[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":                      # 空欄は条件全体。数字なら、そのショット（元ファイル名の先頭 3 桁）だけ
            m &= (df["shot"].to_numpy() == int(r["ショット"])) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


def load(mat, grp, o, plist=None, exclude=None):
    """結合ファイルを読み、形式を確かめ、問題データ一覧の「除外」を除く。列 sens には、「感度確認」の ID が入る（なければ空文字）"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    df = pd.read_csv(path_of(mat, grp, o), encoding="utf-8-sig")
    assert list(df.columns) == list(COLS), f"列名が想定と違う: {list(df.columns)}"
    df = df.rename(columns=COLS)
    assert df.isna().sum().sum() == 0, "欠損がある"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), "ファイル名の形式に合わない行がある"
    assert np.allclose(p[1].astype(float), df["Tset"]) and np.allclose(p[2].astype(float), df["v"]) \
        and np.allclose(p[3].astype(float), df["H"]), "ファイル名と列の値が一致しない"
    df["shot"] = p[0].astype(int)
    df["Tset"] = df["Tset"].astype(float)
    plist = read_problem_list() if plist is None else plist
    n0 = len(df)
    if exclude:
        drop, _ = problem_mask(df, mat, grp, plist, ("除外",))
        df = df[~drop].reset_index(drop=True)
    REMOVED[(mat, grp, o)] = n0 - len(df)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"])
    df["ln_P31"], df["ln_P32"] = np.log(df["P31"] * 1e6), np.log(df["P32"] * 1e6)   # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_v"] = np.log(df["v"])
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)          # 流動先端温度 [K]
    for c in ["gd", "Tff", "P32"]:   # 条件ごとの平均（成形条件から見たばらつきを見るモデル用。cm は condition mean）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    return df


def load_all(exclude=None):
    """全材料・全組・外れ値なし/あり を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp, o): load(mat, grp, o, plist, exclude) for mat in MATS for grp in ["5data", "3data", "Alldata"] for o in ["なし", "あり"]}
    for mat in MATS:
        for o in ["なし", "あり"]:
            assert not (set(out[(mat, "5data", o)]["cond"]) & set(out[(mat, "3data", o)]["cond"])), "学習と検証で条件が重なっている"
    return out


def legacy_check(D_):
    """全条件のファイルから一覧で除いたデータが、1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）と同じ行か（PP の 3data・Alldata）"""
    rows = []
    for (mat, grp), _ in LEGACY_FILES.items():
        for o in ["なし", "あり"]:
            old = pd.read_csv(path_of(mat, grp, o, LEGACY_FILES), encoding="utf-8-sig")
            same = set(old["元ファイル"]) == set(D_[(mat, grp, o)]["file"])
            rows.append({"材料": MATS[mat], "組": grp, "外れ値": o, "一覧で除いた後の行数": len(D_[(mat, grp, o)]), "no1.5-3.0 の行数": len(old), "同じ行": same})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- モデル
def kernel_for(d, linear=False):
    """ARD Matern 5/2 ＋ ノイズ。linear=True なら、線形カーネル（DotProduct）も足す（低速側への外挿のため。CLAUDE.md §5）"""
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def make_gpr(d, linear=False):
    return GaussianProcessRegressor(kernel=kernel_for(d, linear), n_restarts_optimizer=N_RESTARTS, random_state=SEED)


def fit_ols(X_tr, y_tr):
    """切片ありの最小二乗回帰。係数、(XᵀX)⁻¹、残差分散を返す"""
    A_tr = np.column_stack([np.ones(len(X_tr)), X_tr])
    beta, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    resid = y_tr - A_tr @ beta
    return beta, np.linalg.inv(A_tr.T @ A_tr), resid @ resid / (len(y_tr) - A_tr.shape[1])


def predict_ols(model, X):
    """予測平均と、新しい1ショットに対する予測標準偏差"""
    beta, XtX_inv, s2 = model
    A_new = np.column_stack([np.ones(len(X)), X])
    lev = np.einsum("ij,jk,ik->i", A_new, XtX_inv, A_new)
    return A_new @ beta, np.sqrt(s2 * (1.0 + lev))


MODELS = {}             # 記号 → {"kind": "ols" か "gpr", "cols": [入力の列], "linear": 線形カーネルを足すか}。ノートで決める


def fit_model(key, tr):
    spec = MODELS[key]
    X, y = tr[spec["cols"]].to_numpy(), tr["ln_eta"].to_numpy()
    if spec["kind"] == "ols":
        return {"kind": "ols", "model": fit_ols(X, y)}
    sc = StandardScaler().fit(X)              # 標準化と中心化は学習データだけで計算する
    y0 = y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(sc.transform(X), y - y0)
    return {"kind": "gpr", "gp": gp, "sc": sc, "y0": y0}


def predict_model(key, fit, te):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X = te[MODELS[key]["cols"]].to_numpy()
    if fit["kind"] == "ols":
        mu, sd = predict_ols(fit["model"], X)
        return mu, sd, np.full(len(X), np.nan), float(np.sqrt(fit["model"][2]))
    mu, sd = fit["gp"].predict(fit["sc"].transform(X), return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"], sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD（各ショットの予測分布の負の対数密度の平均）、予測標準偏差の平均"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)),
            "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)), "mean_sd": float(np.mean(sd))}


def kernel_row(key, fit):
    """学習した GPR のハイパーパラメータ（振幅、長さスケール、ノイズ、線形項、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    k = gp.kernel_
    lin = MODELS[key].get("linear", False)
    prod, lin_prod = (k.k1.k1, k.k1.k2) if lin else (k.k1, None)
    ls = np.atleast_1d(prod.k2.length_scale)
    row = {"amplitude": float(np.sqrt(prod.k1.constant_value)), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "LML": float(gp.log_marginal_likelihood_value_),
           "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05),
           "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin_prod is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin_prod.k1.constant_value)), "lin_sigma0": float(lin_prod.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(MODELS[key]["cols"], ls)})
    return row


def within_stats(df):
    """同じ条件内のばらつき（プール）。dof：自由度、var_after_gd：ln γ̇ で説明した残りの分散"""
    g = df.groupby("cond")
    dof = int((g.size() - 1).sum())
    de = df["ln_eta"] - g["ln_eta"].transform("mean")
    dg = df["ln_gd"] - g["ln_gd"].transform("mean")
    var_e = float((de ** 2).sum() / dof)
    b = float((de * dg).sum() / (dg ** 2).sum())
    return {"dof": dof, "var_ln_eta": var_e, "sd_ln_eta": var_e ** 0.5, "corr_with_ln_gd": float((de * dg).sum() / np.sqrt((de ** 2).sum() * (dg ** 2).sum())),
            "slope_vs_ln_gd": b, "var_after_gd": float(((de - b * dg) ** 2).sum() / (dof - 1))}


# ---------------------------------------------------------------- データの分け方
def split_V(mat, o):
    """5data で学習し、3data で検証する（射出速度の両端の外挿）。te の group 列は、低速側（3・5 mm/s）か高速側（280・320 mm/s）"""
    tr, te = D[(mat, "5data", o)].copy(), D[(mat, "3data", o)].copy()
    te["group"] = np.where(te["v"] <= 5, "低速側", np.where(te["v"] >= 280, "高速側", "その他"))
    assert (te["group"] != "その他").all(), "検証データに、低速側・高速側のどちらでもない条件がある"
    return tr, te


def split_random(df):
    """行単位のランダム 20%（従来の分け方。random_state = SEED）"""
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    te = df.iloc[idx_te].copy()
    te["group"] = "全体"
    return df.iloc[idx_tr].copy(), te


def split_T(mat, o, t_test=210):
    """設定温度 t_test の全条件をテストにし、5data の残りの温度で学習する（温度の内挿の検証）。3data の同じ温度は te3 として別に返す"""
    d5, d3 = D[(mat, "5data", o)], D[(mat, "3data", o)]
    tr, te = d5[d5["Tset"] != t_test].copy(), d5[d5["Tset"] == t_test].copy()
    te["group"] = "全体"
    te3 = d3[d3["Tset"] == t_test].copy()
    te3["group"] = np.where(te3["v"] <= 5, "低速側", "高速側")
    return tr, te, te3


def pool_of(mat, o):
    """5data と 3data をまとめたデータ"""
    return pd.concat([D[(mat, "5data", o)], D[(mat, "3data", o)]], ignore_index=True)


# ---------------------------------------------------------------- 学習と評価の記録
METRICS, KERNELS, OLSROWS, PRED, FITS = [], [], [], {}, {}      # FITS：(材料, split, データ, モデル) → 学習した予測器
SENS_SUFFIX = "（感度確認の条件を除く）"


def _record(mat, name, data, key, fit, tr, te, pure):
    """学習した予測器 fit で te を予測し、指標（subset ごと）と予測を記録する"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in te["group"].unique():
        if g != "全体":
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    y_te = te["ln_eta"].to_numpy()
    mu, sd, sd_f, noise_sd = predict_model(key, fit, te)
    PRED[(mat, name, data, key)] = pd.DataFrame({"y": y_te, "mu": mu, "sd": sd, "sd_f": sd_f, "v": te["v"].to_numpy(), "H": te["H"].to_numpy(),
                                                 "Tset": te["Tset"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(),
                                                 "is_sens": te["is_sens"].to_numpy()})
    for sub, mask in subsets.items():
        if mask.sum() == 0:
            continue
        ev = evaluate(y_te[mask], mu[mask], sd[mask])
        METRICS.append({"material": MATS[mat], "split": name, "data": data, "model": key, "subset": sub, "n_train": len(tr),
                        "n_test": int(mask.sum()), "n_cond_test": int(te.loc[mask, "cond"].nunique()), **ev,
                        "mean_sd_latent": float(np.nanmean(sd_f[mask])) if fit["kind"] == "gpr" else np.nan, "noise_sd": noise_sd})


def run_split(mat, split, data, tr, te, keys=None, tag="", extra_te=None):
    """全モデルを学習し、指標（subset ごと）・カーネル・予測を記録する。te には group 列がある。
    te に「感度確認」の条件があれば、各 subset から除いた指標（subset 名の後ろに SENS_SUFFIX）も記録する。
    tag は split 名の後ろに付ける印。extra_te（{split 名: テスト}）があれば、同じ学習で、それらも評価する"""
    keys = keys or ALL_KEYS
    name = split + tag
    tests = {name: te, **{s + tag: t for s, t in (extra_te or {}).items()}}
    pure = within_stats(tr)
    for key in keys:
        t0 = time.time()
        fit = fit_model(key, tr)
        FITS[(mat, name, data, key)] = fit
        for nm, te_ in tests.items():
            _record(mat, nm, data, key, fit, tr, te_, pure)
        base = {"material": MATS[mat], "split": name, "data": data, "model": key, "n_train": len(tr)}
        if fit["kind"] == "gpr":
            KERNELS.append({**base, **kernel_row(key, fit), "pure_sd_ln_eta": pure["sd_ln_eta"], "pure_sd_after_gd": pure["var_after_gd"] ** 0.5})
        else:
            names = ["intercept"] + MODELS[key]["cols"]
            OLSROWS.append({**base, "resid_sd": float(np.sqrt(fit["model"][2])), **dict(zip(names, map(float, fit["model"][0])))})
        print(f"{MATS[mat]} {name} {data} {key}: 学習 {len(tr)} 行、{time.time() - t0:.1f} 秒")


# ---------------------------------------------------------------- 条件を1つずつ抜く交差検証（LOCO）
def loco_gpr(key, pool):
    """GPR：ハイパーパラメータを全データで決めて固定し、条件を抜いた予測を閉じた式で計算する（平均 = y_g − [K⁻¹_gg]⁻¹(K⁻¹y)_g、共分散 = [K⁻¹_gg]⁻¹）"""
    spec = MODELS[key]
    X, y = pool[spec["cols"]].to_numpy(), pool["ln_eta"].to_numpy()
    sc = StandardScaler().fit(X)
    Xs, yc = sc.transform(X), y - y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))              # ノイズ込みの共分散行列の逆行列
    alpha = Kinv @ yc
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        Minv = np.linalg.inv(Kinv[np.ix_(idx, idx)])
        mu[idx] = y[idx] - Minv @ alpha[idx]
        sd[idx] = np.sqrt(np.diag(Minv))
    return mu, sd, gp


def loco_ols(key, pool):
    """線形回帰：条件ごとに、その条件を除いて当てはめ直す"""
    cols = MODELS[key]["cols"]
    X, y = pool[cols].to_numpy(), pool["ln_eta"].to_numpy()
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        keep = np.ones(len(y), bool)
        keep[idx] = False
        mu[idx], sd[idx] = predict_ols(fit_ols(X[keep], y[keep]), X[idx])
    return mu, sd


# ---------------------------------------------------------------- 判定の表示（GPR6月ノートの「判定」にならう）
def judge(ok, text):
    """結果の判定を1行で表示する。ok が True なら「OK」、False なら「要確認」"""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{text}")


# ---------------------------------------------------------------- 高解像度のグラフ（GPR6月ノートのセル9-2 にならう。報告書・発表資料用）
HR_COLORS = {"H": {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"},       # 厚みごとの色（6月のノートと同じ並び。黄は、線で見やすい濃さにした）
             "Tset": {190: "#1f77b4", 210: "#2ca02c", 230: "#d62728"}}                       # 設定温度ごとの色
HR_MARKS = {"H": {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}, "Tset": {190: "o", 210: "s", 230: "^"}}
HR_DEFAULT = dict(layouts=["thickness", "temperature"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                  xlim={"thickness": None, "temperature": (30, 50000)}, ylim={"thickness": None, "temperature": (0.5, 5000)}, figsize=(15, 10), dpi=600, font={"axis": 24, "legend": 24, "title": 24, "tick": 28},
                  marker=200, marker_shot=70, line=3.0, show_title=True, show_xlabel=False, show_ylabel=False, formats=["png"], show=False, n_grid=120)


def hr_split_conds(mat, data, split, t_test=210):
    """学習に使った条件の集合。S_V：5data／S_T：5data の t_test 以外の温度／ALL：5data と 3data"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if split == "S_V":
        return set(d5["cond"])
    if split == "S_T":
        return set(d5.loc[d5["Tset"] != t_test, "cond"])
    return set(d5["cond"]) | set(d3["cond"])


def hr_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。Tff・P3_2・射出速度は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（どの入力のモデルでも描けるように）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), tff=("ln_Tff", "mean"), p32=("ln_P32", "mean"), lv=("ln_v", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    return pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": ip("tff"), "ln_Tff_cm": ip("tff"),
                         "ln_P32": ip("p32"), "ln_P32_cm": ip("p32"), "ln_v": ip("lv")})


def hr_flow(predict, mat, data, train_conds, title_model, stem, out_dir, **kw):
    """せん断速度 ― 粘度の学習結果グラフを、高解像度で 1 枚ずつ保存する（6月のノートのセル9-2 と同じ見せ方）。
    predict(Xg) → (予測平均 ln η, 予測標準偏差（ノイズ込み）, 潜在関数の標準偏差)。Xg は hr_inputs の DataFrame。
    train_conds：学習に使った条件の集合（それ以外は、検証の点として白抜きで重ねる）。設定は HR_DEFAULT を参照（kw で上書き）。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**HR_DEFAULT, **kw}
    os.makedirs(out_dir, exist_ok=True)
    pool = pool_of(mat, data)
    curves = {}                                                         # (厚み, 設定温度) → (x, 予測平均, 帯の幅の標準偏差)。レイアウトが違っても同じ線なので、1 回だけ計算する
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf = predict(hr_inputs(sub, lg, h, t))
        curves[(float(h), int(t))] = (np.exp(lg), np.asarray(mu), np.asarray(sd if cfg["band"] == "predict" else sdf))
    saved = []
    ft = cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout in cfg["layouts"]:
        by, sk = ("H", "Tset") if layout == "thickness" else ("Tset", "H")        # by：1 枚ごとに固定する量、sk：1 枚の中で線を分ける量
        for gv in sorted({k[0] if by == "H" else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"], dpi=cfg["dpi"])
            series, any_test, any_sens, xs, ys = [], False, False, [], []
            for sv in sorted({(k[1] if by == "H" else k[0]) for k in curves if (k[0] if by == "H" else k[1]) == gv}):
                h, t = (gv, sv) if by == "H" else (sv, gv)
                col, mk = HR_COLORS[sk][sv], HR_MARKS[sk][sv]
                x, mu, s = curves[(float(h), int(t))]
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.22, zorder=15)
                    ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                    ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                ax.plot(x, np.exp(mu), color=col, lw=cfg["line"], zorder=20)
                sub = pool[(np.isclose(pool["H"], h)) & (pool["Tset"] == t)]
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, tr_m, se_m, msize = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["cond"].isin(train_conds).to_numpy(), a["sens"].to_numpy().astype(bool), cfg["marker"]
                else:
                    px, py, tr_m, se_m, msize = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["cond"].isin(train_conds).to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[tr_m], py[tr_m], s=msize, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14 - list(HR_COLORS[sk]).index(sv))
                if cfg["show_test"] and (~tr_m).any():
                    ax.scatter(px[~tr_m], py[~tr_m], s=msize, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14 - list(HR_COLORS[sk]).index(sv))
                    any_test = True
                if cfg["mark_sens"] and se_m.any():
                    ax.scatter(px[se_m], py[se_m], s=msize * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if sk == "H" else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=col, lw=cfg["line"], marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.6,
                               label=(f"{sv} mm" if sk == "H" else f"{int(sv)} ℃")) for sv, col, mk in series]
            if any_test:
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]), markerfacecolor="white", markeredgewidth=2.5, label="検証の条件（白抜き）"))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]), markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout] if isinstance(cfg["xlim"], dict) else cfg["xlim"]       # 範囲：(下限, 上限)＝固定／None＝その図のデータに合わせる
            yl = cfg["ylim"][layout] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            auto_x, auto_y = xl is None, yl is None
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if auto_x else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if auto_y else yl))
            for axis, auto in ((ax.xaxis, auto_x), (ax.yaxis, auto_y)):
                if auto:                               # 範囲が狭いので、1・2・5 × 10^n に目盛りを付けて、普通の数で表す
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                else:                                  # 6月のノートと同じ（10 の累乗の目盛り）
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv} mm" if by == "H" else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MATS[mat]}　せん断速度と粘度の関係（{gtxt}）\n{title_model}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'by-thickness' if by == 'H' else 'by-temperature'}_{(str(gv) + 'mm') if by == 'H' else (str(int(gv)) + 'C')}"
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=cfg["dpi"])
                saved.append(path)
            if cfg["show"]:
                plt.show()
            plt.close(fig)
    return saved

## 2. データ・モデル・候補の選び方

In [ ]:
# セル2：データの読み込みと、モデルの定義（手順3b と同じ）
D = load_all()


def spec(kind, cols, linear=True):
    return {"kind": kind, "cols": cols, "linear": linear}


BASE = ["ln_gd", "ln_H"]
MODELS.update({
    "B0": spec("gpr", BASE), "B0T": spec("gpr", BASE + ["Tset"]),
    "B0F": spec("gpr", BASE + ["ln_Tff"]), "B0FT": spec("gpr", BASE + ["ln_Tff", "Tset"]),
    "B2": spec("gpr", BASE + ["ln_P32"]), "B2T": spec("gpr", BASE + ["ln_P32", "Tset"]),
    "B2F": spec("gpr", BASE + ["ln_Tff", "ln_P32"]), "B2FT": spec("gpr", BASE + ["ln_Tff", "ln_P32", "Tset"]),
    "R0": spec("gpr", ["ln_gd_cm", "ln_H"]), "R2": spec("gpr", ["ln_v", "ln_H", "Tset"]), "R3": spec("gpr", ["ln_gd_cm", "ln_H", "Tset"]),
    "R3F": spec("gpr", ["ln_gd_cm", "ln_H", "Tset", "ln_Tff_cm"]), "R3P": spec("gpr", ["ln_gd_cm", "ln_H", "Tset", "ln_P32_cm"]),
})
LABELS = {
    "B0": r"B0：［$\dot{\gamma}$, $H$］", "B0T": r"B0T：［$\dot{\gamma}$, $H$, $T_{set}$］",
    "B0F": r"B0F：［$\dot{\gamma}$, $H$, $T_{ff}$］", "B0FT": r"B0FT：［$\dot{\gamma}$, $H$, $T_{ff}$, $T_{set}$］",
    "B2": r"B2：［$\dot{\gamma}$, $H$, P3_2］", "B2T": r"B2T：［$\dot{\gamma}$, $H$, P3_2, $T_{set}$］",
    "B2F": r"B2F：［$\dot{\gamma}$, $H$, $T_{ff}$, P3_2］", "B2FT": r"B2FT：［$\dot{\gamma}$, $H$, $T_{ff}$, P3_2, $T_{set}$］",
    "R0": r"R0：［$\dot{\gamma}$（条件平均）, $H$］", "R2": r"R2：［$v$, $H$, $T_{set}$］", "R3": r"R3：［$\dot{\gamma}$（条件平均）, $H$, $T_{set}$］",
    "R3F": r"R3F：［$\dot{\gamma}$（条件平均）, $H$, $T_{set}$, $T_{ff}$（条件平均）］", "R3P": r"R3P：［$\dot{\gamma}$（条件平均）, $H$, $T_{set}$, P3_2（条件平均）］",
}
INPUT_JA = {"ln_gd": r"$\ln\dot{\gamma}$", "ln_H": "ln H", "ln_Tff": "ln Tff", "ln_P32": "ln P3_2", "Tset": "Tset", "ln_v": "ln v", "ln_gd_cm": r"$\ln\dot{\gamma}$（条件平均）",
            "ln_Tff_cm": "ln Tff（条件平均）", "ln_P32_cm": "ln P3_2（条件平均）"}
SPLIT_JA = {"S_V": "S_V（5data → 3data）", "S_T": f"S_T（190・230 ℃ → {TEMP_HOLDOUT} ℃）"}
for mat in MATS:
    tr, te = D[(mat, "5data", MAIN_DATA)], D[(mat, "3data", MAIN_DATA)]
    print(f"{MATS[mat]}：学習（5data）{len(tr)} 行 {tr['cond'].nunique()} 条件、検証（3data）{len(te)} 行 {te['cond'].nunique()} 条件")

In [ ]:
# セル3：候補のモデルを決める（手順3b の S_V の NLPD が良い順。B0T は必ず入れる）
def pick_candidates():
    if CANDIDATES is not None:
        return {m: list(v) for m, v in CANDIDATES.items()}, "設定エリアで指定"
    path = os.path.join(STEP3B_DIR, "metrics.csv")
    if not os.path.exists(path):
        return {m: CAND_FIXED + ["R2"] for m in MATS}, "手順3b の結果がないので、B0T・R3・R2"
    mt = pd.read_csv(path)
    out = {}
    for mat in MATS:
        t = mt[(mt["material"] == MATS[mat]) & (mt["split"] == "S_V") & (mt["data"] == MAIN_DATA) & (mt["model"].isin(CAND_POOL)) & mt["subset"].isin(["低速側", "高速側" + SENS_SUFFIX])]
        score = t.pivot_table(index="model", columns="subset", values="NLPD").mean(axis=1).drop(CAND_FIXED, errors="ignore").sort_values()
        out[mat] = CAND_FIXED + list(score.index[:N_CAND_EXTRA])
    return out, f"必ず入れる {CAND_FIXED} と、手順3b の S_V の NLPD（低速側と、高速側から感度確認の条件を除いたものの平均）が良い {N_CAND_EXTRA} 個"


CAND, CAND_HOW = pick_candidates()
for mat, keys in CAND.items():
    assert all(k in MODELS for k in keys), f"未定義のモデルがある: {keys}"
    print(f"{MATS[mat]}：{keys}")
print("選び方：", CAND_HOW)
pd.DataFrame([{"material": MATS[m], "model": k, "how": CAND_HOW} for m, ks in CAND.items() for k in ks]).to_csv(os.path.join(OUT_DIR, "candidates.csv"), index=False, encoding="utf-8-sig")

## 3. サンプラーと予測の関数

ハイパーパラメータは対数で扱う。θ は scikit-learn の `kernel.theta`（固定していないハイパーパラメータの対数）と同じ順番：[ln 振幅², ln 長さスケール…, ln 線形項の振幅², ln σ₀, ln ノイズ分散]。

In [ ]:
# セル4：MCMC（適応型ランダムウォーク・メトロポリス法）、予測、評価の関数
TARGET_ACC = 0.234


def theta_names(key):
    return ["log_c"] + [f"log_ls_{c}" for c in MODELS[key]["cols"]] + ["log_c_lin", "log_sigma0", "log_noise"]


PARAM_JA = {"log_c": "振幅²", "log_c_lin": "線形項の振幅²", "log_sigma0": "線形項の σ₀", "log_noise": "ノイズ分散"}


def param_ja(name):
    if name.startswith("log_ls_"):
        return "長さスケール（" + INPUT_JA[name[len("log_ls_"):]] + "）"
    return PARAM_JA[name]


def run_chain(X_tr, y_tr, d, theta0, n_warm, n_samp, seed):
    """適応型ランダムウォーク・メトロポリス法の1チェーン。
    調整中：提案分布 N(θ, λ·Σ) の Σ を足跡から学び（前半75%）、λ を採択率が TARGET_ACC に近づくよう合わせる。本番：Σ と λ を固定。"""
    rng = np.random.default_rng(seed)
    gp = GaussianProcessRegressor(kernel=kernel_for(d, True), optimizer=None).fit(X_tr, y_tr)
    lo, hi = gp.kernel_.bounds[:, 0], gp.kernel_.bounds[:, 1]

    def logpost(th):
        if np.any(th <= lo) or np.any(th >= hi):
            return -np.inf                       # 事前分布の範囲の外は確率0
        return gp.log_marginal_likelihood(th)    # 範囲内では一様なので、対数周辺尤度そのもの

    k = len(theta0)
    th = np.array(theta0, dtype=float)
    lp = logpost(th)
    cov, log_lam = np.eye(k) * 0.05 ** 2, 0.0
    warm = np.empty((n_warm, k))
    draws, lps, acc = np.empty((n_samp, k)), np.empty(n_samp), np.zeros(n_warm + n_samp, dtype=bool)
    t0 = time.time()
    for it in range(n_warm + n_samp):
        L = np.linalg.cholesky(np.exp(log_lam) * cov)
        prop = th + L @ rng.standard_normal(k)
        lp_prop = logpost(prop)
        a = np.exp(min(0.0, lp_prop - lp)) if np.isfinite(lp_prop) else 0.0
        if rng.uniform() < a:
            th, lp, acc[it] = prop, lp_prop, True
        if it < n_warm:                           # 調整（本番では行わない）
            warm[it] = th
            log_lam += (it + 1) ** -0.6 * (a - TARGET_ACC)
            if it >= 199 and (it + 1) % 50 == 0 and it < 0.75 * n_warm:
                cov = np.cov(warm[(it + 1) // 2: it + 1].T) * (2.38 ** 2 / k) + np.eye(k) * 1e-10
        else:
            draws[it - n_warm], lps[it - n_warm] = th, lp
    return {"draws": draws, "lp": lps, "accepted": acc[n_warm:], "acc_warm": acc[:n_warm].mean(),
            "acc_samp": acc[n_warm:].mean(), "lam": np.exp(log_lam), "seconds": time.time() - t0}


def gp_predict_thetas(X_tr, y_tr, X_new, d, thetas):
    """θ ごとの GPR の予測平均と標準偏差（ノイズ込み）。形は (θ の数, 点の数)"""
    base = kernel_for(d, True)
    mus, sds = [], []
    for th in np.atleast_2d(thetas):
        gp = GaussianProcessRegressor(kernel=base.clone_with_theta(th), optimizer=None).fit(X_tr, y_tr)
        mu, sd = gp.predict(X_new, return_std=True)
        mus.append(mu)
        sds.append(sd)
    return np.array(mus), np.array(sds)


def evaluate_mixture(y, mus, sds):
    """正規分布を等しい重みで混ぜた予測分布での指標。mus, sds は (成分数, 点の数)。成分が 1 つなら evaluate と同じ"""
    mus, sds = np.atleast_2d(mus), np.atleast_2d(sds)
    mu = mus.mean(axis=0)
    cdf = norm.cdf((y - mus) / sds).mean(axis=0)
    log_dens = logsumexp(norm.logpdf(y, mus, sds), axis=0) - np.log(len(mus))
    sd_mix = np.sqrt((sds ** 2 + mus ** 2).mean(axis=0) - mu ** 2)
    return {"R2": float(r2_score(y, mu)) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean((cdf >= 0.025) & (cdf <= 0.975))), "NLPD": float(-np.mean(log_dens)), "mean_sd": float(np.mean(sd_mix))}, mu, sd_mix


def to_arrays(trace, cols):
    t = trace.sort_values(["chain", "draw"])
    n_chain = t["chain"].nunique()
    return {c: t[c].to_numpy().reshape(n_chain, -1) for c in cols}


def diag_of(trace, cols):
    """最大の R-hat と、最小の有効サンプル数（bulk）"""
    arrs = to_arrays(trace, cols)
    diag = az.summary(az.from_dict({"posterior": arrs}), kind="diagnostics", round_to="none")
    return float(diag["r_hat"].max()), float(diag["ess_bulk"].min())


print("関数を定義した")

## 4. 学習と MCMC によるサンプリング

- 最尤推定（手順3b と同じ設定の `fit_model`）で θ の初期値を決める。4 本のチェーンの初期値は、最尤推定値に、ノイズ以外だけ N(0, 0.3²) のずれを加えたもの。
- 本番のサンプル（対数スケール）は `results/step3c_mcmc/posterior_<分割>_<材料>_<モデル>.csv` に、チェーンごとの採択率などは `chain_info.csv` に保存する。

In [ ]:
# セル5：学習データの用意と、サンプリング（保存済みのサンプルがあれば読み込む）
def get_split(mat, split):
    if split == "S_V":
        tr, te = split_V(mat, MAIN_DATA)
        return tr, {"S_V": te}
    tr, te, te3 = split_T(mat, MAIN_DATA, TEMP_HOLDOUT)
    return tr, {"S_T": te, "S_T3": te3}


def prepare(mat, split, key):
    """学習データだけで標準化・中心化した入力と目的変数"""
    tr, tests = get_split(mat, split)
    spec_ = MODELS[key]
    sc = StandardScaler().fit(tr[spec_["cols"]].to_numpy())
    y0 = tr["ln_eta"].mean()
    return tr, tests, sc, y0, sc.transform(tr[spec_["cols"]].to_numpy()), tr["ln_eta"].to_numpy() - y0


info_path = os.path.join(OUT_DIR, "chain_info.csv")
chain_info = pd.read_csv(info_path) if os.path.exists(info_path) else pd.DataFrame()
combos = [(mat, split, key) for mat in MATS for split in ["S_V", "S_T"] for key in CAND[mat]]
PREP = {c: prepare(*c) for c in combos}                           # (tr, tests, sc, y0, Xs, yc)
t_all = time.time()


def fit_job(key, tr):
    warnings.simplefilter("ignore")                  # 並列の子プロセスでも、下限への張り付きの警告を出さない
    return fit_model(key, tr)


fit_list = Parallel(n_jobs=N_JOBS)(delayed(fit_job)(key, PREP[(mat, split, key)][0]) for (mat, split, key) in combos)    # 最尤推定（並列）
MLE = dict(zip(combos, fit_list))
print(f"最尤推定：{len(combos)} 個、{(time.time() - t_all) / 60:.1f} 分")


def post_path(combo):
    mat, split, key = combo
    return os.path.join(OUT_DIR, f"posterior_{split}_{mat}_{key}.csv")


def starts_seeds(idx, combo):
    """チェーンの初期値（最尤推定値に、ノイズ以外だけ N(0, 0.3²) のずれを加える）とシード"""
    mat, split, key = combo
    d = PREP[combo][4].shape[1]
    cols = theta_names(key)
    bnd = kernel_for(d, True).bounds
    th0 = np.clip(MLE[combo]["gp"].kernel_.theta, bnd[:, 0] + 1e-6, bnd[:, 1] - 1e-6)
    assert len(th0) == len(cols), "θ の長さが想定と違う"
    rng = np.random.default_rng(SEED + 100 * idx)
    shift = np.r_[np.ones(len(cols) - 1), 0.0]
    starts = [np.clip(th0 + rng.normal(0, 0.3, len(cols)) * shift, bnd[:, 0] + 1e-6, bnd[:, 1] - 1e-6) for _ in range(CHAINS)]
    return starts, [SEED + 1000 * idx + c for c in range(CHAINS)]


traces = {}
for combo in combos:                                              # 保存済みのサンプルがあれば読み込む
    if USE_SAVED_SAMPLES and os.path.exists(post_path(combo)):
        saved = pd.read_csv(post_path(combo))
        per_chain = saved.groupby("chain").size()
        if len(per_chain) == CHAINS and (per_chain >= N_SAMP).all():
            traces[combo] = saved
            print(f"{MATS[combo[0]]} {combo[1]} {combo[2]}: 保存済みのサンプルを読み込み（1チェーン {int(per_chain.iloc[0])} 回）")

pending, n_samp = [c for c in combos if c not in traces], N_SAMP
while pending:
    tasks = []
    for combo in pending:
        starts, seeds = starts_seeds(combos.index(combo), combo)
        _, _, _, _, Xs, yc = PREP[combo]
        tasks += [(combo, c, Xs, yc, starts[c], seeds[c]) for c in range(CHAINS)]
    t0 = time.time()
    outs = Parallel(n_jobs=N_JOBS)(delayed(run_chain)(Xs, yc, Xs.shape[1], st, N_WARM, n_samp, sd) for (_, _, Xs, yc, st, sd) in tasks)
    print(f"本番 {n_samp:,}回 × {CHAINS}チェーン × {len(pending)} 組：{(time.time() - t0) / 60:.1f} 分")
    still = []
    for combo in pending:
        mat, split, key = combo
        res = [o for (t, o) in zip(tasks, outs) if t[0] == combo]
        cols = theta_names(key)
        trace = pd.concat([pd.DataFrame({"chain": c, "draw": np.arange(n_samp), **{col: r["draws"][:, j] for j, col in enumerate(cols)},
                                         "lp": r["lp"], "accepted": r["accepted"].astype(int)}) for c, r in enumerate(res)], ignore_index=True)
        rhat, ess = diag_of(trace, cols)
        print(f"  {MATS[mat]} {split} {key}: 最大 R-hat {rhat:.3f}、最小 ESS(bulk) {ess:.0f}、採択率 " + ", ".join(f"{r['acc_samp']:.2f}" for r in res))
        if (rhat > 1.01 or ess < 400) and n_samp < N_SAMP_MAX:
            still.append(combo)                                       # R-hat が大きい、または有効サンプル数が少ないので、本番を増やしてやり直す
            continue
        trace.to_csv(post_path(combo), index=False, float_format="%.7g")
        traces[combo] = pd.read_csv(post_path(combo))                 # 保存した値（7桁）で以降を計算する
        rows = pd.DataFrame([{"material": MATS[mat], "split": split, "model": key, "chain": c, "acc_warm": r["acc_warm"], "acc_samp": r["acc_samp"],
                              "lambda": r["lam"], "chain_seconds": r["seconds"], "wall_seconds": max(x["seconds"] for x in res)} for c, r in enumerate(res)])
        if len(chain_info):
            chain_info = chain_info[~((chain_info["material"] == MATS[mat]) & (chain_info["split"] == split) & (chain_info["model"] == key))]
        chain_info = pd.concat([chain_info, rows], ignore_index=True)
        chain_info.to_csv(info_path, index=False)
    pending, n_samp = still, min(int(n_samp * 2.5), N_SAMP_MAX)

RUN_INFO = {k: {"chains": int(t["chain"].nunique()), "n_warm": N_WARM, "n_samp_per_chain": int(len(t) // t["chain"].nunique()), "draws_total": int(len(t))}
            for k, t in traces.items()}
run_df = pd.DataFrame([{"material": MATS[m], "split": s, "model": k, **info} for (m, s, k), info in RUN_INFO.items()])
print(f"\n合計 {(time.time() - t_all) / 60:.1f} 分")
print(run_df.to_string(index=False))

## 5. 収束診断と事後分布

- R-hat（1.01 以下が目安）と有効サンプル数 ESS（bulk・tail。400 以上が目安）は ArviZ で計算する。
- 事後分布の要約は、振幅（√振幅²）・長さスケール・σ₀・ノイズ分散の元の単位で示す（長さスケールは標準化後の入力に対する値）。MLE は最尤推定値。

In [ ]:
# セル6：収束診断と事後分布の要約
rows = []
for (mat, split, key), trace in traces.items():
    cols = theta_names(key)
    arrs = to_arrays(trace, cols)
    diag = az.summary(az.from_dict({"posterior": arrs}), kind="diagnostics", round_to="none")
    th_mle = MLE[(mat, split, key)]["gp"].kernel_.theta
    bnd = kernel_for(len(MODELS[key]["cols"]), True).bounds
    for j, col in enumerate(cols):
        v = np.exp(arrs[col].ravel())
        label, mle, lo_b, hi_b = col.replace("log_", ""), np.exp(th_mle[j]), np.exp(bnd[j, 0]), np.exp(bnd[j, 1])
        if col in ("log_c", "log_c_lin"):                                  # 振幅² → 振幅
            v, mle, lo_b, hi_b = np.sqrt(v), np.sqrt(mle), np.sqrt(lo_b), np.sqrt(hi_b)
            label = "amplitude" if col == "log_c" else "lin_amplitude"
        q = np.percentile(v, [2.5, 50, 97.5])
        rows.append({"material": MATS[mat], "split": split, "model": key, **RUN_INFO[(mat, split, key)], "param": label, "MLE": mle,
                     "post_mean": v.mean(), "post_sd": v.std(ddof=1), "q2.5": q[0], "median": q[1], "q97.5": q[2],
                     "r_hat": diag.loc[col, "r_hat"], "ess_bulk": diag.loc[col, "ess_bulk"], "ess_tail": diag.loc[col, "ess_tail"],
                     "post_at_bound": bool(q[0] < lo_b * 1.05 or q[2] > hi_b * 0.95)})
post_df = pd.DataFrame(rows)
post_df.to_csv(os.path.join(OUT_DIR, "posterior_summary.csv"), index=False, encoding="utf-8-sig")
print(post_df[["material", "split", "model", "param", "MLE", "median", "q2.5", "q97.5", "r_hat", "ess_bulk", "post_at_bound"]].to_string(index=False, float_format=lambda x: f"{x:.4g}"))

stat_rows = []
for (mat, split, key), trace in traces.items():
    ci = chain_info[(chain_info["material"] == MATS[mat]) & (chain_info["split"] == split) & (chain_info["model"] == key)]
    p = post_df[(post_df["material"] == MATS[mat]) & (post_df["split"] == split) & (post_df["model"] == key)]
    stat_rows.append({"material": MATS[mat], "split": split, "model": key, **RUN_INFO[(mat, split, key)],
                      "acc_samp_min": ci["acc_samp"].min(), "acc_samp_max": ci["acc_samp"].max(), "sampling_min": ci["wall_seconds"].iloc[0] / 60,
                      "max_r_hat": p["r_hat"].max(), "min_ess_bulk": p["ess_bulk"].min(), "min_ess_tail": p["ess_tail"].min(), "n_post_at_bound": int(p["post_at_bound"].sum())})
stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(os.path.join(OUT_DIR, "sampler_stats.csv"), index=False, encoding="utf-8-sig")
print("\n【サンプラーの統計】")
print(stat_df.round(3).to_string(index=False))
n_bad = int(((stat_df["max_r_hat"] > 1.01) | (stat_df["min_ess_bulk"] < 400)).sum())
judge(n_bad == 0, f"全 {len(stat_df)} 組で R-hat ≤ 1.01、ESS(bulk) ≥ 400（満たさないのは {n_bad} 組。満たさないなら、N_SAMP を増やして再実行）")
n_bnd = int((stat_df["n_post_at_bound"] > 0).sum())
judge(n_bnd == 0, f"事後分布の 95% 区間が、事前分布の範囲の端（下限・上限の 5% 以内）に届いたモデルは {n_bnd} 組（届いたら、そのハイパーパラメータは、データだけでは決まらない）")

In [ ]:
# セル7：トレース図（左：チェーンごとの分布、右：本番の推移。値は log10）。S_V の学習のものを材料×モデルごとに保存する
CHAIN_COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
for (mat, split, key), trace in traces.items():
    cols = theta_names(key)
    fig, axes = plt.subplots(len(cols), 2, figsize=(12, 1.9 * len(cols)), gridspec_kw={"width_ratios": [1, 3]})
    for i, col in enumerate(cols):
        for ch, t in trace.groupby("chain"):
            v = t[col].to_numpy() / np.log(10)
            axes[i, 0].hist(v, bins=40, histtype="step", density=True, color=CHAIN_COLORS[ch % 4], label=f"チェーン {ch + 1}")
            axes[i, 1].plot(t["draw"], v, lw=0.3, alpha=0.7, color=CHAIN_COLORS[ch % 4])
        axes[i, 0].set_ylabel(f"log10 {param_ja(col)}", fontsize=8)
        axes[i, 1].set_ylabel(f"log10 {param_ja(col)}", fontsize=8)
        axes[i, 0].xaxis.set_major_locator(MaxNLocator(nbins=4))
    axes[0, 0].legend(fontsize=7)
    axes[-1, 0].set_xlabel("値（log10）")
    axes[-1, 1].set_xlabel("本番の反復回数")
    r = RUN_INFO[(mat, split, key)]
    fig.suptitle(f"トレース図：{MATS[mat]}　{LABELS[key]}、{SPLIT_JA[split]}\n本番 {r['n_samp_per_chain']:,}回 × {r['chains']}チェーン（調整 {r['n_warm']:,}回）", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig_trace_{split}_{mat}_{key}.png"), dpi=FIG_DPI)
    plt.close(fig)
print("トレース図を保存した（fig_trace_*.png）")

In [ ]:
# セル8：事後分布と最尤推定値（赤の破線）、事前分布の範囲の端（灰色の点線）。材料×分割ごとに 1 枚
for mat in MATS:
    for split in ["S_V", "S_T"]:
        keys = CAND[mat]
        n_col = max(len(theta_names(k)) for k in keys)
        fig, axes = plt.subplots(len(keys), n_col, figsize=(2.9 * n_col, 2.5 * len(keys)), squeeze=False)
        for i, key in enumerate(keys):
            cols, th_mle = theta_names(key), MLE[(mat, split, key)]["gp"].kernel_.theta
            bnd = kernel_for(len(MODELS[key]["cols"]), True).bounds
            for j in range(n_col):
                ax = axes[i, j]
                if j >= len(cols):
                    ax.axis("off")
                    continue
                v = traces[(mat, split, key)][cols[j]].to_numpy() / np.log(10)
                ax.hist(v, bins=50, density=True, color="tab:blue", alpha=0.6, label="事後分布（MCMC）")
                ax.axvline(th_mle[j] / np.log(10), color="red", ls="--", lw=1.2, label="最尤推定値")
                for b in bnd[j] / np.log(10):
                    if v.min() - 0.3 < b < v.max() + 0.3:
                        ax.axvline(b, color="gray", ls=":", lw=1.2, label="事前分布の範囲の端")
                ax.set_title(f"{key}：{param_ja(cols[j])}", fontsize=8)
                ax.tick_params(labelsize=7)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
                if i == 0 and j == 0:
                    ax.legend(fontsize=6)
        fig.suptitle(f"事後分布（MCMC）と最尤推定値：{MATS[mat]}、{SPLIT_JA[split]}（横軸は log10）", fontsize=11)
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"fig_posterior_vs_mle_{split}_{mat}.png"), dpi=FIG_DPI)
        plt.show()

## 6. 予測と評価（最尤推定と MCMC の比較）

- **MCMC**：事後サンプルを等間隔に N_PRED_DRAWS 個選び、各サンプルの GPR の予測分布（ノイズ込み）を混ぜる。予測平均は混合の平均、95%区間の被覆率は混合分布の累積分布、NLPD は混合分布の密度で計算する。
- **MLE**：最尤推定値 1 つで予測する（手順3b と同じ）。
- `mean_sd` は、テストデータでの予測標準偏差（ln η）の平均。予測区間の広さの目安。
- 「（感度確認の条件を除く）」は、検証の P03（PP の 190 ℃・0.5 mm・320 mm/s）を評価から除いた指標。

In [ ]:
# セル9：予測と評価
res_rows, PREDS = [], {}
t_all = time.time()
for (mat, split, key), trace in traces.items():
    tr, tests, sc, y0, Xs, yc = prepare(mat, split, key)
    d = Xs.shape[1]
    tr_ = trace.sort_values(["chain", "draw"])
    idx = np.linspace(0, len(tr_) - 1, N_PRED_DRAWS).round().astype(int)
    thetas = tr_[theta_names(key)].to_numpy()[idx]
    th_mle = MLE[(mat, split, key)]["gp"].kernel_.theta
    for tname, te in tests.items():
        Xt = sc.transform(te[MODELS[key]["cols"]].to_numpy())
        y = te["ln_eta"].to_numpy()
        mus, sds = gp_predict_thetas(Xs, yc, Xt, d, thetas)
        mu1, sd1 = gp_predict_thetas(Xs, yc, Xt, d, th_mle)
        subsets = {"全体": np.ones(len(te), bool)}
        if "group" in te.columns:
            for g in te["group"].unique():
                if g != "全体":
                    subsets[g] = (te["group"] == g).to_numpy()
        if te["is_sens"].any():
            subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
        for method, (m_, s_) in {"MCMC": (mus + y0, sds), "MLE": (mu1 + y0, sd1)}.items():
            for sub, mask in subsets.items():
                if mask.sum() == 0:
                    continue
                ev, _, _ = evaluate_mixture(y[mask], m_[:, mask], s_[:, mask])
                res_rows.append({"material": MATS[mat], "split": tname, "model": key, "method": method, "subset": sub, "n_test": int(mask.sum()), **ev})
            ev_all, mu_mix, sd_mix = evaluate_mixture(y, m_, s_)
            PREDS[(mat, tname, key, method)] = pd.DataFrame({"y": y, "mu": mu_mix, "sd": sd_mix, "H": te["H"].to_numpy(), "v": te["v"].to_numpy(), "Tset": te["Tset"].to_numpy(),
                                                             "group": te["group"].to_numpy(), "is_sens": te["is_sens"].to_numpy()})
    print(f"{MATS[mat]} {split} {key}: 予測 完了")
mcmc_df = pd.DataFrame(res_rows)
mcmc_df.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding="utf-8-sig")
print(f"\n合計 {(time.time() - t_all) / 60:.1f} 分")


def show_cmp(mat, tname, subs, cols=("RMSE", "coverage95", "NLPD", "mean_sd")):
    t = mcmc_df[(mcmc_df["material"] == MATS[mat]) & (mcmc_df["split"] == tname) & mcmc_df["subset"].isin(subs)]
    p = t.pivot_table(index=["model", "method"], columns="subset", values=list(cols))
    p = p.reindex(columns=[(c, s) for c in cols for s in subs if (c, s) in p.columns])
    print(p.round(3).to_string())


for mat in MATS:
    print(f"\n【S_V】{MATS[mat]}")
    show_cmp(mat, "S_V", ["低速側", "高速側", "高速側" + SENS_SUFFIX])
    print(f"\n【S_T】{MATS[mat]}（{TEMP_HOLDOUT} ℃ をテスト）")
    show_cmp(mat, "S_T", ["全体"])
    show_cmp(mat, "S_T3", ["全体"])

In [ ]:
# セル10：MLE と MCMC の差（被覆率・NLPD・区間の広さ）の判定
diffs = []
for mat in MATS:
    for tname, sub in [("S_V", "低速側"), ("S_V", "高速側"), ("S_T", "全体"), ("S_T3", "全体")]:
        for key in CAND[mat]:
            a = mcmc_df[(mcmc_df["material"] == MATS[mat]) & (mcmc_df["split"] == tname) & (mcmc_df["model"] == key) & (mcmc_df["subset"] == sub)].set_index("method")
            if len(a) < 2:
                continue
            diffs.append({"material": MATS[mat], "split": tname, "subset": sub, "model": key,
                          "RMSE_MLE": a.loc["MLE", "RMSE"], "RMSE_MCMC": a.loc["MCMC", "RMSE"],
                          "cover_MLE": a.loc["MLE", "coverage95"], "cover_MCMC": a.loc["MCMC", "coverage95"],
                          "NLPD_MLE": a.loc["MLE", "NLPD"], "NLPD_MCMC": a.loc["MCMC", "NLPD"],
                          "sd_MLE": a.loc["MLE", "mean_sd"], "sd_MCMC": a.loc["MCMC", "mean_sd"],
                          "NLPD_diff": a.loc["MCMC", "NLPD"] - a.loc["MLE", "NLPD"], "sd_ratio": a.loc["MCMC", "mean_sd"] / a.loc["MLE", "mean_sd"]})
diff_df = pd.DataFrame(diffs)
diff_df.to_csv(os.path.join(OUT_DIR, "mle_vs_mcmc.csv"), index=False, encoding="utf-8-sig")
print(diff_df.round(3).to_string(index=False))
for mat in MATS:
    s = diff_df[diff_df["material"] == MATS[mat]]
    big = s[s["NLPD_diff"].abs() >= 0.2]
    judge(big.empty, f"{MATS[mat]}：MCMC と最尤推定の NLPD の差が、すべて 0.2 未満（最大 {s['NLPD_diff'].abs().max():.2f}）。差が小さければ、最尤推定で十分。0.2 以上のもの：{[f'{a} {b} {c}（MCMC − 最尤 = {d:+.2f}）' for a, b, c, d in zip(big['split'], big['subset'], big['model'], big['NLPD_diff'])] or 'なし'}。負の大きな値は、最尤推定が局所解に止まっていた（MCMC のほうが良い）ことを表す")
    wide = s[~s["sd_ratio"].between(0.8, 1.5)]
    judge(wide.empty, f"{MATS[mat]}：MCMC の予測区間の広さ（平均の標準偏差）が、最尤推定の 0.8〜1.5 倍（範囲 {s['sd_ratio'].min():.2f}〜{s['sd_ratio'].max():.2f}）。外れたもの：{[f'{a} {b} {c}（{d:.2f} 倍）' for a, b, c, d in zip(wide['split'], wide['subset'], wide['model'], wide['sd_ratio'])] or 'なし'}")

## 7. 図

- 図 A：最尤推定と MCMC の、被覆率・NLPD・予測区間の広さの比較
- 図 B：設定温度の効果（190/230 ℃ の粘度比）の事後分布。ハイパーパラメータの不確かさによる、粘度比のばらつき
- 図 C：R3・B0T の予測曲線（S_V の学習）。MCMC の混合予測の 95% 区間と、最尤推定の 95% 区間

In [ ]:
# セル11：図A 最尤推定と MCMC の比較
PANELS = [("S_V", "低速側"), ("S_V", "高速側"), ("S_T", "全体"), ("S_T3", "全体")]
fig, axes = plt.subplots(3, len(MATS), figsize=(15, 12))
for j, mat in enumerate(MATS):
    for i, (col, ylab) in enumerate([("coverage95", "95%区間の被覆率"), ("NLPD", "NLPD（小さいほど良い）"), ("mean_sd", "予測区間の広さ（標準偏差の平均）")]):
        ax = axes[i, j]
        items = [(p, k) for p in PANELS for k in CAND[mat]]
        x = np.arange(len(items))
        for method, off, colr in [("MLE", -0.2, "tab:red"), ("MCMC", 0.2, "tab:blue")]:
            vals = []
            for (tname, sub), key in items:
                a = mcmc_df[(mcmc_df["material"] == MATS[mat]) & (mcmc_df["split"] == tname) & (mcmc_df["model"] == key) & (mcmc_df["subset"] == sub) & (mcmc_df["method"] == method)]
                vals.append(float(a[col].iloc[0]) if len(a) else np.nan)
            ax.bar(x + off, vals, 0.4, color=colr, label="最尤推定" if method == "MLE" else "MCMC")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{t}{'' if s == '全体' else chr(10) + s}\n{k}" for (t, s), k in items], fontsize=6.5)
        ax.set_title(f"{MATS[mat]}：{ylab}")
        if col == "coverage95":
            ax.axhline(0.95, color="k", ls="--", lw=0.8)
        ax.grid(alpha=0.3, axis="y")
        if i == 0 and j == 0:
            ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figA_mle_vs_mcmc.png"), dpi=FIG_DPI)
plt.show()

In [ ]:
# セル12：図B 設定温度の効果（Tset だけを 190 → 230 ℃）の事後分布
TSET_CAND = {mat: [k for k in CAND[mat] if "Tset" in MODELS[k]["cols"]] for mat in MATS}
eff_rows = []
fig, axes = plt.subplots(len(MATS), 4, figsize=(18, 4.6 * len(MATS)), squeeze=False)
for i, mat in enumerate(MATS):
    pool = pool_of(mat, MAIN_DATA)
    base = pool[(pool["Tset"] == 210) & ~pool["is_sens"]]
    obs = None
    for j, h in enumerate([0.5, 1.0, 1.5, 2.0]):
        ax = axes[i, j]
        g = base[base["H"] == h]
        data_, labels_ = [], []
        for key in TSET_CAND[mat]:
            tr, tests, sc, y0, Xs, yc = prepare(mat, "S_V", key)
            trace = traces[(mat, "S_V", key)].sort_values(["chain", "draw"])
            idx = np.linspace(0, len(trace) - 1, N_PRED_DRAWS).round().astype(int)
            thetas = trace[theta_names(key)].to_numpy()[idx]
            lo_, hi_ = g.copy(), g.copy()
            lo_["Tset"], hi_["Tset"] = 190.0, 230.0
            m_lo, _ = gp_predict_thetas(Xs, yc, sc.transform(lo_[MODELS[key]["cols"]].to_numpy()), Xs.shape[1], thetas)
            m_hi, _ = gp_predict_thetas(Xs, yc, sc.transform(hi_[MODELS[key]["cols"]].to_numpy()), Xs.shape[1], thetas)
            ratio = np.exp(np.median(m_lo - m_hi, axis=1))                                  # θ ごとの、210 ℃ のショットの粘度比の中央値
            f1 = MLE[(mat, "S_V", key)]
            r_mle = float(np.exp(np.median(predict_model(key, f1, lo_)[0] - predict_model(key, f1, hi_)[0])))
            eff_rows.append({"material": MATS[mat], "H": h, "model": key, "ratio_mle": r_mle, "ratio_mcmc_median": float(np.median(ratio)),
                             "ratio_q2.5": float(np.percentile(ratio, 2.5)), "ratio_q97.5": float(np.percentile(ratio, 97.5)), "share_gt1": float(np.mean(ratio > 1))})
            data_.append(ratio)
            labels_.append(key)
        if data_:
            ax.violinplot(data_, showmedians=True)
            ax.set_xticks(np.arange(1, len(labels_) + 1))
            ax.set_xticklabels(labels_)
            ax.axhline(1.0, color="gray", lw=0.8)
        ax.set_title(f"{MATS[mat]}　厚み {h} mm", fontsize=10)
        ax.set_ylabel("粘度の比（190 ℃ ÷ 230 ℃）")
        ax.grid(alpha=0.3, axis="y")
fig.suptitle("設定温度の効果（Tset だけを 190 → 230 ℃）の事後分布：ハイパーパラメータの不確かさ（S_V の学習）", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "figB_tset_effect_posterior.png"), dpi=FIG_DPI)
plt.show()
eff_df = pd.DataFrame(eff_rows)
eff_df.to_csv(os.path.join(OUT_DIR, "tset_effect_mcmc.csv"), index=False, encoding="utf-8-sig")
print(eff_df.round(3).to_string(index=False))
for mat in MATS:
    e = eff_df[eff_df["material"] == MATS[mat]]
    judge(bool((e["share_gt1"] > 0.95).all()), f"{MATS[mat]}：どの厚み・モデルでも、事後サンプルの 95% 以上で、190 ℃ のほうが 230 ℃ より粘い（最小 {e['share_gt1'].min():.2f}）")

In [ ]:
# セル13：図C 予測曲線（S_V の学習。MCMC と最尤推定の 95% 区間）。R3（条件平均の γ̇）と B0T（ショットごとの γ̇）の 2 枚
T_COL = {190: "tab:blue", 210: "tab:green", 230: "tab:red"}
H_LEVELS = [0.5, 1.0, 1.5, 2.0]


def curve_fig(key):
    fig, axes = plt.subplots(len(H_LEVELS), len(MATS), figsize=(6.5 * len(MATS), 14), squeeze=False)
    for j, mat in enumerate(MATS):
        tr, tests, sc, y0, Xs, yc = prepare(mat, "S_V", key)
        trace = traces[(mat, "S_V", key)].sort_values(["chain", "draw"])
        idx = np.linspace(0, len(trace) - 1, N_PRED_DRAWS).round().astype(int)
        thetas = trace[theta_names(key)].to_numpy()[idx]
        allp = pool_of(mat, MAIN_DATA)
        te_cond = set(D[(mat, "3data", MAIN_DATA)]["cond"])
        for i, h in enumerate(H_LEVELS):
            ax = axes[i, j]
            for t, col in T_COL.items():
                sub = allp[(allp["H"] == h) & (allp["Tset"] == t)]
                if sub.empty:
                    continue
                lg = np.linspace(sub["ln_gd"].min(), sub["ln_gd"].max(), 100)
                Xg = pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t)})
                Xn = sc.transform(Xg[MODELS[key]["cols"]].to_numpy())
                mus, sds = gp_predict_thetas(Xs, yc, Xn, Xs.shape[1], thetas)
                _, mu, sd = evaluate_mixture(np.zeros(len(lg)), mus + y0, sds)
                mu1, sd1, _, _ = predict_model(key, MLE[(mat, "S_V", key)], Xg)
                ax.plot(np.exp(lg), np.exp(mu), color=col, lw=1.4)
                ax.fill_between(np.exp(lg), np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd), color=col, alpha=0.15)
                ax.plot(np.exp(lg), np.exp(mu1 - 1.96 * sd1), color=col, lw=0.7, ls="--")
                ax.plot(np.exp(lg), np.exp(mu1 + 1.96 * sd1), color=col, lw=0.7, ls="--")
                cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"))
                is_te = np.array([c in te_cond for c in cm.index])
                ax.scatter(np.exp(cm["lg"][~is_te]), np.exp(cm["le"][~is_te]), s=14, color=col, alpha=0.8, zorder=3)
                ax.scatter(np.exp(cm["lg"][is_te]), np.exp(cm["le"][is_te]), s=26, facecolors="none", edgecolors=col, marker="s", linewidths=1.2, zorder=3)
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(alpha=0.3, which="both")
            ax.set_ylabel(f"厚み {h} mm\n" + r"$\eta$ [Pa·s]")
            if i == 0:
                ax.set_title(f"{MATS[mat]}：{key}（帯：MCMC の 95% 区間、破線：最尤推定の 95% 区間）", fontsize=9)
            if i == len(H_LEVELS) - 1:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ [1/s]")
    handles = [plt.Line2D([], [], color=c, label=f"{t} ℃") for t, c in T_COL.items()]
    handles += [plt.Line2D([], [], color="gray", marker="o", ls="", label="学習の条件平均"), plt.Line2D([], [], color="gray", marker="s", mfc="none", ls="", label="検証の条件平均")]
    axes[0, 0].legend(handles=handles, fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"figC_{key}_curves.png"), dpi=FIG_DPI)
    plt.show()


for key_ in ["R3", "B0T"]:
    curve_fig(key_)

## 8. 判定のまとめ

In [ ]:
# セル14：保存したファイルと、判定のまとめ
print("【判定のまとめ】（上の各節の判定を、観点ごとにもう一度）")
judge(n_bad == 0, f"収束：R-hat ≤ 1.01、ESS(bulk) ≥ 400（満たさないのは {n_bad} 組）")
judge(n_bnd == 0, f"事前分布の範囲の端に届いたモデルは {n_bnd} 組")
for mat in MATS:
    s = diff_df[diff_df["material"] == MATS[mat]]
    judge(bool((s["NLPD_diff"].abs() < 0.2).all()), f"{MATS[mat]}：MCMC と最尤推定の NLPD の差 最大 {s['NLPD_diff'].abs().max():.2f}（0.2 未満なら、最尤推定で十分。大きいのは、最尤推定が局所解に止まったモデル）")
print("\n保存したファイル:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", os.path.normpath(os.path.join(OUT_DIR, f)))

## 9. 高解像度のグラフ（報告書・発表資料用）

せん断速度 ― 粘度の学習結果グラフを、GPR6月ノートのセル9-2 と同じ見せ方で、**600 dpi（15 × 10 インチ）**で 1 枚ずつ保存する。線は MCMC の予測の平均（事後サンプルの予測を混ぜた分布）、帯は 95% 区間、塗りつぶしの点は学習に使った条件、白抜きの点は学習に使わなかった条件（検証）、× は問題データ一覧の「感度確認」の条件。

- **レイアウト**：`"thickness"`＝厚みごとに 1 枚（3 つの設定温度の曲線）、`"temperature"`＝設定温度ごとに 1 枚（4 つの厚みの曲線。6月のノートと同じ）。
- **変えられるもの**（セルの先頭の設定エリア）：描くモデル・材料・学習の種類、帯（予測区間／信頼区間）、点（条件平均／ショットごと）、軸の範囲、予測線の延長、図の大きさ・解像度・文字・マーカーの大きさ、タイトルと軸名の表示、保存形式（png／pdf／svg）。
- 全体を一度実行して、学習が終わった後なら、このセルだけを何度でも実行し直せる。

In [ ]:
# セル15：高解像度のグラフ（せん断速度 ― 粘度。MCMC の予測。GPR6月ノートのセル9-2 にならう）
# ==========================================
# ▼▼ 設定エリア（グラフの見た目・文字サイズ。このセルの設定だけを直して、このセルだけを実行し直せる） ▼▼
# ==========================================
HR_MODELS = ["R3"]                 # 描くモデル（このノートで MCMC をかけたモデル。例：["R3", "B0T", "R2"]。記号の意味は、ノート冒頭の表）
HR_SPLIT = "S_V"                   # どの学習のモデルか："S_V"（5data で学習）／"S_T"（190・230 ℃ で学習）。MCMC のサンプルがあるものだけ
HR_MATERIALS = ["PP", "MIX"]       # 描く材料（PP：PP、MIX：粉砕材混合 PP）
HR_LAYOUTS = ["thickness", "temperature"]   # "thickness"：厚みごとに 1 枚（3 つの設定温度）／"temperature"：設定温度ごとに 1 枚（4 つの厚み。6月のノートと同じ）
HR_BAND = "predict"                # 帯："predict"＝95% 予測区間（ショット間のばらつき込み）／"latent"＝95% 信頼区間（平均の不確かさだけ）／None＝帯なし
HR_POINTS = "cond"                 # 点："cond"＝条件平均／"shot"＝ショットごと
HR_SHOW_TEST = True                # 学習に使わなかった条件（検証）を、白抜きで重ねるか
HR_MARK_SENS = True                # 問題データ一覧の「感度確認」の条件を、× で重ねるか
HR_X_RANGE = None                  # 予測線の範囲 [1/s]。None なら、各線のデータの範囲を HR_EXTEND 倍に延長。(30, 50000) のように指定すると、全部の線をその範囲に延長（外挿の確認用）
HR_EXTEND = 1.3
HR_XLIM = {"thickness": None, "temperature": (30, 50000)}   # 横軸（せん断速度 [1/s]）の表示範囲。(下限, 上限)＝固定（10 の累乗の目盛り。6月と同じ）／None＝その図のデータに合わせる（1・2・5 の目盛り）。
HR_YLIM = {"thickness": None, "temperature": (0.5, 5000)}   # 縦軸（粘度 [Pa·s]）の表示範囲。同じ書き方。レイアウトごとに変えられる（全部を同じ範囲にするなら、(30, 50000) のように書く）
HR_FIGSIZE = (15, 10)              # 図の大きさ（幅, 高さ）[インチ]
HR_DPI = 600                       # 解像度
HR_FONT = {"axis": 24, "legend": 24, "title": 24, "tick": 28}   # 軸名・凡例・タイトル・目盛り数字の文字サイズ
HR_MARKER = 200                    # 点の大きさ（条件平均のとき。面積）
HR_MARKER_SHOT = 70                # 点の大きさ（ショットごとのとき）
HR_LINE = 3.0                      # 予測線の太さ
HR_SHOW_TITLE = True               # タイトルを付けるか
HR_SHOW_XLABEL = False             # 軸名を付けるか（6月のノートと同じく、既定は付けない。資料に貼るとき、別に付ける場合のため）
HR_SHOW_YLABEL = False
HR_FORMATS = ["png"]               # 保存形式。"png"・"pdf"・"svg" から選ぶ（pdf・svg は、拡大しても粗くならない）
HR_SHOW = False                    # True なら、画面にも表示する（重い）
# ==========================================
HR_KW = dict(layouts=HR_LAYOUTS, band=HR_BAND, points=HR_POINTS, show_test=HR_SHOW_TEST, mark_sens=HR_MARK_SENS, x_range=HR_X_RANGE, extend=HR_EXTEND,
             xlim=HR_XLIM, ylim=HR_YLIM, figsize=HR_FIGSIZE, dpi=HR_DPI, font=HR_FONT, marker=HR_MARKER, marker_shot=HR_MARKER_SHOT, line=HR_LINE,
             show_title=HR_SHOW_TITLE, show_xlabel=HR_SHOW_XLABEL, show_ylabel=HR_SHOW_YLABEL, formats=HR_FORMATS, show=HR_SHOW)
HR_DIR = os.path.join(OUT_DIR, "highres")

HR_SPLIT_JA = {"S_V": "5data で学習", "S_T": "190・230 ℃ で学習"}


def hr_predict_mcmc(mat, split, key):
    """MCMC の事後サンプル（N_PRED_DRAWS 個）で予測した分布を等しい重みで混ぜた、予測の平均・標準偏差（ノイズ込み）・標準偏差（ノイズを除く）"""
    _, _, sc, y0, Xs, yc = PREP[(mat, split, key)]
    trace = traces[(mat, split, key)].sort_values(["chain", "draw"])
    idx = np.linspace(0, len(trace) - 1, N_PRED_DRAWS).round().astype(int)
    thetas = trace[theta_names(key)].to_numpy()[idx]
    noise = np.exp(thetas[:, -1])                                         # θ の最後は ln ノイズ分散

    def predict(Xg):
        Xn = sc.transform(Xg[MODELS[key]["cols"]].to_numpy())
        mus, sds = gp_predict_thetas(Xs, yc, Xn, Xs.shape[1], thetas)
        mus = mus + y0
        mu = mus.mean(axis=0)
        sd = np.sqrt((sds ** 2 + mus ** 2).mean(axis=0) - mu ** 2)
        sdf = np.sqrt((np.maximum(sds ** 2 - noise[:, None], 0.0) + mus ** 2).mean(axis=0) - mu ** 2)
        return mu, sd, sdf
    return predict


hr_saved = []
for mat in HR_MATERIALS:
    for key in HR_MODELS:
        if (mat, HR_SPLIT, key) not in traces:
            print(f"{MATS[mat]} {HR_SPLIT} {key}：MCMC のサンプルがないので、描かない（上の候補のモデルを指定する）")
            continue
        t0 = time.time()
        hr_saved += hr_flow(hr_predict_mcmc(mat, HR_SPLIT, key), mat, MAIN_DATA, hr_split_conds(mat, MAIN_DATA, HR_SPLIT, TEMP_HOLDOUT),
                            f"{LABELS[key]}（{HR_SPLIT_JA[HR_SPLIT]}、MCMC）", f"hr_{mat}_{key}_{HR_SPLIT}_mcmc", HR_DIR, **HR_KW)
        print(f"{MATS[mat]} {key}：保存したファイル 累計 {len(hr_saved)} 個（{time.time() - t0:.0f} 秒）")
judge(len(hr_saved) > 0, f"高解像度のグラフを {len(hr_saved)} 個保存した（{os.path.normpath(HR_DIR)}）。形式 {HR_FORMATS}、{HR_DPI} dpi")